# 02. Phase 1: ベースライン分析（人間の特徴量設計）

**役割分担のうち「人間」のパート。** EDA で得た史実ベースの仮説を特徴量に落とし込み、
ロジスティック回帰・決定木・ランダムフォレストの 3 モデルを 5-fold CV で比較する。

| 段階 | 特徴量 | 最高 CV Accuracy |
|---|---|---|
| v1 (基本整形) | Pclass/Sex/Age/SibSp/Parch/Embarked/cabin_missing/family_size | ~0.80 |
| v2 (人間が改善) | + Fare 復活 / Title / 相互作用項 / Fare_per_person | **0.8204** |

In [ ]:
import sys; sys.path.insert(0, '../src')
import pandas as pd
from feature_engineering import build_features_v1, build_features_v2, V1_FEATURES, feature_columns
from model_training import build_baseline_models
from evaluation import make_cv, evaluate_models

train = pd.read_csv('../data/train.csv')
y = train['Survived']
cv = make_cv(5)

## v1: 指示通りのデータ整形
- Age: Pclass 別中央値で補填
- Cabin: `cabin_missing` フラグ化 → 列削除
- Embarked: 最頻値で補填
- `family_size = SibSp + Parch + 1`
- Sex/Embarked を数値化、PassengerId/Ticket/Name/Fare を削除

In [ ]:
train_v1 = build_features_v1(train, train)
X1 = train_v1[V1_FEATURES]
res_v1 = evaluate_models(build_baseline_models(), X1, y, cv)
print('=== Phase 1 v1: 結果表 (5-fold CV) ===')
res_v1

## 結果分析と改善 (v2)
v1 は 3 モデルが 0.79〜0.80 で横並び → **モデルではなく特徴量の情報量が頭打ち**。
そこで人間のドメイン知識で特徴を追加する:
- **Fare 復活**: 経済力の連続値。Pclass の粗いカテゴリを補完。
- **Title**: 名前の敬称に年齢・性別・地位が凝縮。欠損だらけの Age の代理。
- **相互作用項 Sex×Pclass / Age×Pclass**: 「上流階級の女性・子供」を明示。
- **Fare_per_person / IsAlone**: 家族割の補正と単身フラグ。

In [ ]:
train_v2 = build_features_v2(train, train)
feats2 = feature_columns(train_v2)
X2 = train_v2[feats2]
res_v2 = evaluate_models(build_baseline_models(), X2, y, cv)
print('=== Phase 1 v2: 人間が改善した特徴量 (5-fold CV) ===')
res_v2

## Phase 1 のまとめ
- 人間のドメイン知識による特徴量設計で **0.80 → 0.8204** へ改善。
- ここまでが「人間の思考」で到達できる水準。
- 次は `03_v2_optimization.ipynb` で **自動最適化ループ** (GridSearch / XGBoost / LightGBM / アンサンブル) に引き継ぐ。